In [ ]:
import os
os.makedirs('/tmp/l3', exist_ok=True)
open('/tmp/l3/build_occ_dataset.py','w',encoding='utf-8').write('"""Stage F data: BEV occupancy inputs and labels for all 404 nuScenes-mini keyframes (ego frame, 200x200 cells of 0.5 m, rows +x, cols +y, +/-50 m).\n\nINPUT (single keyframe only, no sweeps):\n  LiDAR channels : [log1p(#points with 0.3<z<3.0)/3, (max z +1)/5 where obstacle-height points exist, log1p(#all points)/3, mean intensity/255]\n  Radar channels : 5 radars, devkit filter (invalid_state==0, ambig_state==3): [log1p(#hits)/2, mean rcs/50, mean |v_comp|/20]\nLABEL (never seen by the model as input):\n  2 object : rasterised footprints of ALL annotated boxes of the 10 evaluated categories (real human GT)\n  1 static : cell hit by obstacle-height points (0.3<z<3.0 m, ego z=0 is ground) in >= 4 of the 11 ego-motion-compensated sweeps\n             (keyframe + up to 10 previous sweeps), and not on/next to an object footprint  (PSEUDO-GT, derived from LiDAR)\n  0 free   : everything else (includes cells never observed: occluded / out of range - the label cannot tell them apart)\nAll thresholds are fixed here, before any model is trained."""\nimport glob, json, os, sys\nimport numpy as np\nfrom scipy.ndimage import binary_dilation\n\nOUT = sys.argv[1]\nos.makedirs(OUT, exist_ok=True)\nV = None\nfor r, d, f in os.walk(\'/kaggle/input\'):\n    if \'v1.0-mini\' in d:\n        V = r; break\nROOT = V; TAB = os.path.join(ROOT, \'v1.0-mini\')\ntab = lambda n: json.load(open(os.path.join(TAB, n + \'.json\')))\nsample = {s[\'token\']: s for s in tab(\'sample\')}; scene = {s[\'token\']: s for s in tab(\'scene\')}\nsd = {d[\'token\']: d for d in tab(\'sample_data\')}; cs = {c[\'token\']: c for c in tab(\'calibrated_sensor\')}; ego = {e[\'token\']: e for e in tab(\'ego_pose\')}\ninst = {i[\'token\']: i for i in tab(\'instance\')}; cat = {c[\'token\']: c[\'name\'] for c in tab(\'category\')}\nann = {}\nfor a in tab(\'sample_annotation\'):\n    ann.setdefault(a[\'sample_token\'], []).append(a)\nfrom nuscenes.utils import splits\nGEN = {\'vehicle.car\', \'vehicle.truck\', \'vehicle.bus.bendy\', \'vehicle.bus.rigid\', \'vehicle.trailer\', \'vehicle.construction\', \'vehicle.motorcycle\', \'vehicle.bicycle\',\n       \'human.pedestrian.adult\', \'human.pedestrian.child\', \'human.pedestrian.construction_worker\', \'human.pedestrian.police_officer\', \'movable_object.trafficcone\', \'movable_object.barrier\'}\nNC, RES, EXT = 200, 0.5, 50.0\nBODY = np.array([[-1.0, 3.7], [-1.1, 1.1]])\nTY = {(\'F\', 4): \'<f4\', (\'F\', 8): \'<f8\', (\'I\', 1): \'i1\', (\'I\', 2): \'<i2\', (\'I\', 4): \'<i4\', (\'U\', 1): \'u1\', (\'U\', 2): \'<u2\', (\'U\', 4): \'<u4\'}\n\n\ndef qR(q):\n    w, x, y, z = q\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])\n\n\ndef read_pcd(p):\n    with open(p, \'rb\') as f:\n        h = {}\n        while True:\n            k, *v = f.readline().decode().strip().split(); h[k] = v\n            if k == \'DATA\':\n                break\n        dt = np.dtype([(n, TY[(t, int(s))]) for n, t, s in zip(h[\'FIELDS\'], h[\'TYPE\'], h[\'SIZE\'])])\n        return np.frombuffer(f.read(), dtype=dt, count=int(h[\'POINTS\'][0]))\n\n\ndef cells(xy):\n    r = np.floor((xy[:, 0] + EXT) / RES).astype(int); c = np.floor((xy[:, 1] + EXT) / RES).astype(int)\n    ok = (r >= 0) & (r < NC) & (c >= 0) & (c < NC)\n    return r[ok], c[ok], ok\n\n\ndef lidar_to_ego(sdr):\n    c = cs[sdr[\'calibrated_sensor_token\']]\n    raw = np.fromfile(os.path.join(ROOT, sdr[\'filename\']), dtype=np.float32).reshape(-1, 5)\n    return (qR(c[\'rotation\']) @ raw[:, :3].T).T + np.array(c[\'translation\']), raw[:, 3]\n\n\ndef strip_body(P, *extra):\n    m = ~((P[:, 0] > BODY[0, 0]) & (P[:, 0] < BODY[0, 1]) & (P[:, 1] > BODY[1, 0]) & (P[:, 1] < BODY[1, 1]))\n    return (P[m],) + tuple(e[m] for e in extra)\n\n\nxs = -EXT + (np.arange(NC) + 0.5) * RES; XX, YY = np.meshgrid(xs, xs, indexing=\'ij\')\n\n\ndef raster_boxes(boxes):\n    m = np.zeros((NC, NC), bool)\n    for cx, cy, w, l, yaw in boxes:\n        c, s = np.cos(yaw), np.sin(yaw); r0 = max(0, int((cx - l - w + EXT) / RES) - 2); r1 = min(NC, int((cx + l + w + EXT) / RES) + 3)\n        c0 = max(0, int((cy - l - w + EXT) / RES) - 2); c1 = min(NC, int((cy + l + w + EXT) / RES) + 3)\n        if r0 >= r1 or c0 >= c1:\n            continue\n        dx, dy = XX[r0:r1, c0:c1] - cx, YY[r0:r1, c0:c1] - cy\n        u = c * dx + s * dy; v = -s * dx + c * dy\n        m[r0:r1, c0:c1] |= (np.abs(u) <= l / 2) & (np.abs(v) <= w / 2)\n    return m\n\n\nmini_train, mini_val = set(splits.mini_train), set(splits.mini_val)\nitems = []\nfor t, s in sample.items():\n    nm = scene[s[\'scene_token\']][\'name\']\n    if nm in mini_train or nm in mini_val:\n        items.append((s[\'scene_token\'], s[\'timestamp\'], t, \'train\' if nm in mini_train else \'val\', nm))\nitems.sort()\nkey_lidar = {d[\'sample_token\']: d for d in sd.values() if d[\'is_key_frame\'] and \'/LIDAR_TOP/\' in d[\'filename\']}\nkey_radar = {}\nfor d in sd.values():\n    if d[\'is_key_frame\'] and \'/RADAR_\' in d[\'filename\']:\n        key_radar.setdefault(d[\'sample_token\'], []).append(d)\n\nN = len(items); XL = np.zeros((N, 4, NC, NC), np.float16); XR = np.zeros((N, 3, NC, NC), np.float16); Y = np.zeros((N, NC, NC), np.uint8)\nSTATS = []\nfor k, (_, ts, t, split, nm) in enumerate(items):\n    kl = key_lidar[t]; ek = ego[kl[\'ego_pose_token\']]; Rk, tk = qR(ek[\'rotation\']), np.array(ek[\'translation\'])\n    P, inten = lidar_to_ego(kl); P, inten = strip_body(P, inten)\n    obst = (P[:, 2] > 0.3) & (P[:, 2] < 3.0)\n    r, c, ok = cells(P[obst, :2]); cnt = np.zeros((NC, NC)); np.add.at(cnt, (r, c), 1)\n    mz = np.full((NC, NC), -np.inf); np.maximum.at(mz, (r, c), P[obst, 2][ok])\n    ra, ca, oka = cells(P[:, :2]); call = np.zeros((NC, NC)); np.add.at(call, (ra, ca), 1); isum = np.zeros((NC, NC)); np.add.at(isum, (ra, ca), inten[oka])\n    XL[k, 0] = np.log1p(cnt) / 3; XL[k, 1] = np.where(np.isfinite(mz), (mz + 1) / 5, 0); XL[k, 2] = np.log1p(call) / 3; XL[k, 3] = np.where(call > 0, isum / np.maximum(call, 1) / 255, 0)\n    # radar\n    hits = np.zeros((NC, NC)); rcs = np.zeros((NC, NC)); spd = np.zeros((NC, NC))\n    for rd in key_radar[t]:\n        a = read_pcd(os.path.join(ROOT, rd[\'filename\'])); a = a[(a[\'invalid_state\'] == 0) & (a[\'ambig_state\'] == 3)]\n        if len(a) == 0:\n            continue\n        cr = cs[rd[\'calibrated_sensor_token\']]; Pe = (qR(cr[\'rotation\']) @ np.stack([a[\'x\'], a[\'y\'], a[\'z\']]).astype(float)).T + np.array(cr[\'translation\'])\n        rr, cc, okr = cells(Pe[:, :2]); np.add.at(hits, (rr, cc), 1); np.add.at(rcs, (rr, cc), a[\'rcs\'][okr]); np.add.at(spd, (rr, cc), np.hypot(a[\'vx_comp\'], a[\'vy_comp\'])[okr])\n    XR[k, 0] = np.log1p(hits) / 2; XR[k, 1] = np.where(hits > 0, rcs / np.maximum(hits, 1) / 50, 0); XR[k, 2] = np.where(hits > 0, spd / np.maximum(hits, 1) / 20, 0)\n    # object labels (human GT)\n    boxes = []\n    for a in ann.get(t, []):\n        if cat[inst[a[\'instance_token\']][\'category_token\']] not in GEN:\n            continue\n        cen = Rk.T @ (np.array(a[\'translation\']) - tk); Rb = Rk.T @ qR(a[\'rotation\'])\n        boxes.append((cen[0], cen[1], a[\'size\'][0], a[\'size\'][1], float(np.arctan2(Rb[1, 0], Rb[0, 0]))))\n    obj = raster_boxes(boxes)\n    # static pseudo-GT: persistence over 11 ego-motion-compensated sweeps\n    persist = (cnt > 0).astype(int); token = kl[\'prev\']; nsw = 0\n    while token and nsw < 10:\n        sw = sd[token]; ep = ego[sw[\'ego_pose_token\']]\n        Pw, _ = lidar_to_ego(sw); Pg = (qR(ep[\'rotation\']) @ Pw.T).T + np.array(ep[\'translation\']); Pk = (Rk.T @ (Pg - tk).T).T\n        Pk, = strip_body(Pk); m = (Pk[:, 2] > 0.3) & (Pk[:, 2] < 3.0); rr, cc, _ = cells(Pk[m, :2]); h = np.zeros((NC, NC), bool); h[rr, cc] = True\n        persist += h; token = sw[\'prev\']; nsw += 1\n    static = (persist >= 4) & ~binary_dilation(obj, np.ones((3, 3), bool))\n    Y[k][static] = 1; Y[k][obj] = 2\n    STATS.append(dict(idx=k, token=t, split=split, scene=nm, sweeps=nsw, n_obj=len(boxes), obj_cells=int(obj.sum()), static_cells=int(static.sum())))\n    if k % 50 == 0:\n        print(\'built\', k, \'/\', N, flush=True)\nsplit = np.array([s[\'split\'] for s in STATS]); tokens = np.array([s[\'token\'] for s in STATS]); scenes = np.array([s[\'scene\'] for s in STATS])\nnp.savez_compressed(os.path.join(OUT, \'occ_dataset.npz\'), XL=XL, XR=XR, Y=Y, split=split, token=tokens, scene=scenes)\njson.dump(STATS, open(os.path.join(OUT, \'occ_dataset_stats.json\'), \'w\'))\ntr, va = split == \'train\', split == \'val\'\nprint(\'train\', tr.sum(), \'val\', va.sum(), \'| label frequency train free/static/object:\', [round(float((Y[tr] == c).mean()), 4) for c in range(3)], \'val:\', [round(float((Y[va] == c).mean()), 4) for c in range(3)])\nprint(\'mean sweeps used\', np.mean([s[\'sweeps\'] for s in STATS]), \'| DATASET BUILD PASS\')\n')
open('/tmp/l3/build_flow_dataset.py','w',encoding='utf-8').write('"""Stage G data: temporal geometry + flow / future labels for the 404 keyframes of Stage F (same sample order, taken from occ_dataset.npz).\n\nFor every keyframe t (ego frame E_t, 200x200 cells of 0.5 m, rows +x, cols +y):\n  H[t,k]      row index of keyframe t-(k+1) (k=0..2); when the scene history is shorter, the oldest available keyframe is reused.\n  A[t,k], b[t,k]  planar rigid map current-cell (x,y) -> location in the history frame:  q = A @ [x,y] + b   (A = (R_p^T R_c)[:2,:2], b = (R_p^T (t_c - t_p))[:2]).\n  FLOW  (2,200,200) m/s : GROUND velocity of the annotated object covering the cell, expressed in E_t axes (static objects ~ 0; not relative to ego). NaN velocity -> not valid.\n  FVALID (200,200)      : cells with a valid flow label.\n  FUT05 / FUT10         : footprints of all annotated objects at t+0.5 s / t+1.0 s (next keyframes) in E_t (world-anchored), AV05 / AV10 availability.\n  FM05 / FM10           : same, restricted to objects moving (speed >= 0.3 m/s) at that future time.\nVelocity of an annotation = finite difference of the instance\'s translation over its previous/next keyframe annotations (the nuScenes devkit rule, max gap 1.5 s)."""\nimport json, os, sys\nimport numpy as np\n\nOUT = sys.argv[1]\nNPZ = \'/tmp/l3/data/occ_dataset.npz\'\nV = None\nfor r, d, f in os.walk(\'/kaggle/input\'):\n    if \'v1.0-mini\' in d:\n        V = r; break\nprint(\'occ dataset:\', NPZ, \'| nuScenes root:\', V)\ntok = np.load(NPZ, allow_pickle=True)[\'token\']\nTAB = os.path.join(V, \'v1.0-mini\'); tab = lambda n: json.load(open(os.path.join(TAB, n + \'.json\')))\nsample = {s[\'token\']: s for s in tab(\'sample\')}; sd = tab(\'sample_data\'); ego = {e[\'token\']: e for e in tab(\'ego_pose\')}\ninst = {i[\'token\']: i for i in tab(\'instance\')}; cat = {c[\'token\']: c[\'name\'] for c in tab(\'category\')}\nannt = {a[\'token\']: a for a in tab(\'sample_annotation\')}; ann_by_sample = {}\nfor a in annt.values():\n    ann_by_sample.setdefault(a[\'sample_token\'], []).append(a)\nkey_lidar = {d[\'sample_token\']: d for d in sd if d[\'is_key_frame\'] and \'/LIDAR_TOP/\' in d[\'filename\']}\nGEN = {\'vehicle.car\', \'vehicle.truck\', \'vehicle.bus.bendy\', \'vehicle.bus.rigid\', \'vehicle.trailer\', \'vehicle.construction\', \'vehicle.motorcycle\', \'vehicle.bicycle\',\n       \'human.pedestrian.adult\', \'human.pedestrian.child\', \'human.pedestrian.construction_worker\', \'human.pedestrian.police_officer\', \'movable_object.trafficcone\', \'movable_object.barrier\'}\nNC, RES, EXT = 200, 0.5, 50.0\nxs = -EXT + (np.arange(NC) + 0.5) * RES; XX, YY = np.meshgrid(xs, xs, indexing=\'ij\')\n\n\ndef qR(q):\n    w, x, y, z = q\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])\n\n\ndef velocity(a):\n    """global xy velocity (m/s) of annotation a, NaN if undefined (devkit rule)."""\n    t0 = sample[a[\'sample_token\']][\'timestamp\'] / 1e6\n    prev = annt[a[\'prev\']] if a[\'prev\'] else None; nxt = annt[a[\'next\']] if a[\'next\'] else None\n    p0 = np.array(a[\'translation\'])\n    if prev and nxt:\n        dt = (sample[nxt[\'sample_token\']][\'timestamp\'] - sample[prev[\'sample_token\']][\'timestamp\']) / 1e6; d = np.array(nxt[\'translation\']) - np.array(prev[\'translation\'])\n    elif nxt:\n        dt = (sample[nxt[\'sample_token\']][\'timestamp\'] / 1e6 - t0); d = np.array(nxt[\'translation\']) - p0\n    elif prev:\n        dt = (t0 - sample[prev[\'sample_token\']][\'timestamp\'] / 1e6); d = p0 - np.array(prev[\'translation\'])\n    else:\n        return np.full(2, np.nan)\n    if dt <= 0 or dt > 1.5:\n        return np.full(2, np.nan)\n    return (d / dt)[:2]\n\n\ndef raster(boxes, values=None):\n    """boxes: (cx,cy,w,l,yaw); values: optional per-box vectors -> (mask, value map)."""\n    m = np.zeros((NC, NC), bool); vm = None if values is None else np.zeros((len(values[0]) if len(values) else 2, NC, NC), np.float32)\n    for i, (cx, cy, w, l, yaw) in enumerate(boxes):\n        c, s = np.cos(yaw), np.sin(yaw); r0 = max(0, int((cx - l - w + EXT) / RES) - 2); r1 = min(NC, int((cx + l + w + EXT) / RES) + 3)\n        c0 = max(0, int((cy - l - w + EXT) / RES) - 2); c1 = min(NC, int((cy + l + w + EXT) / RES) + 3)\n        if r0 >= r1 or c0 >= c1:\n            continue\n        dx, dy = XX[r0:r1, c0:c1] - cx, YY[r0:r1, c0:c1] - cy\n        inside = (np.abs(c * dx + s * dy) <= l / 2) & (np.abs(-s * dx + c * dy) <= w / 2)\n        m[r0:r1, c0:c1] |= inside\n        if values is not None:\n            for ch in range(vm.shape[0]):\n                vm[ch, r0:r1, c0:c1][inside] = values[i][ch]\n    return m, vm\n\n\ndef boxes_at(sample_token, Rc, tc, want_velocity=False):\n    """annotated objects of a keyframe expressed in the ego frame (Rc, tc) of ANOTHER keyframe."""\n    out, vel = [], []\n    for a in ann_by_sample.get(sample_token, []):\n        if cat[inst[a[\'instance_token\']][\'category_token\']] not in GEN:\n            continue\n        cen = Rc.T @ (np.array(a[\'translation\']) - tc); Rb = Rc.T @ qR(a[\'rotation\'])\n        out.append((cen[0], cen[1], a[\'size\'][0], a[\'size\'][1], float(np.arctan2(Rb[1, 0], Rb[0, 0]))))\n        v = velocity(a); vel.append((Rc.T @ np.array([v[0], v[1], 0.0]))[:2] if np.all(np.isfinite(v)) else np.full(2, np.nan))\n    return out, vel\n\n\nN = len(tok); row = {t: i for i, t in enumerate(tok)}\nH = np.zeros((N, 3), np.int64); A = np.zeros((N, 3, 2, 2), np.float32); B = np.zeros((N, 3, 2), np.float32)\nFLOW = np.zeros((N, 2, NC, NC), np.float16); FVALID = np.zeros((N, NC, NC), bool)\nFUT = {h: np.zeros((N, NC, NC), bool) for h in (\'05\', \'10\')}; FM = {h: np.zeros((N, NC, NC), bool) for h in (\'05\', \'10\')}; AV = {h: np.zeros(N, bool) for h in (\'05\', \'10\')}\npose = {}\nfor t in tok:\n    e = ego[key_lidar[t][\'ego_pose_token\']]; pose[t] = (qR(e[\'rotation\']), np.array(e[\'translation\']))\nfor i, t in enumerate(tok):\n    Rc, tc = pose[t]\n    chain = []; cur = t\n    for _ in range(3):\n        p = sample[cur][\'prev\']\n        if not p:\n            break\n        chain.append(p); cur = p\n    for k in range(3):\n        src = chain[k] if k < len(chain) else (chain[-1] if chain else t)\n        H[i, k] = row[src]; Rp, tp = pose[src]\n        A[i, k] = (Rp.T @ Rc)[:2, :2]; B[i, k] = (Rp.T @ (tc - tp))[:2]\n    boxes, vel = boxes_at(t, Rc, tc)\n    m, vm = raster(boxes, vel)\n    fv = m & np.isfinite(vm[0]) & np.isfinite(vm[1]); FLOW[i] = np.where(fv[None], np.nan_to_num(vm), 0); FVALID[i] = fv\n    nxt = t\n    for h in (\'05\', \'10\'):\n        nxt = sample[nxt][\'next\'] if nxt else \'\'\n        if not nxt:\n            continue\n        bx, bv = boxes_at(nxt, Rc, tc); FUT[h][i], _ = raster(bx); AV[h][i] = True\n        moving = [b for b, v in zip(bx, bv) if np.all(np.isfinite(v)) and np.hypot(*v) >= 0.3]\n        FM[h][i], _ = raster(moving)\n    if i % 100 == 0:\n        print(\'built\', i, \'/\', N, flush=True)\nnp.savez_compressed(os.path.join(OUT, \'flow_dataset.npz\'), H=H, A=A, B=B, FLOW=FLOW, FVALID=FVALID, FUT05=FUT[\'05\'], FUT10=FUT[\'10\'], FM05=FM[\'05\'], FM10=FM[\'10\'], AV05=AV[\'05\'], AV10=AV[\'10\'])\nsp = np.hypot(FLOW[:, 0].astype(np.float32), FLOW[:, 1].astype(np.float32))\nprint(\'flow-valid cells / object cells:\', int(FVALID.sum()), \'| moving cells (>=0.3 m/s):\', int(((sp >= 0.3) & FVALID).sum()), \'| median speed of moving cells\', float(np.median(sp[(sp >= 0.3) & FVALID])))\nprint(\'future available 0.5 s:\', int(AV[\'05\'].sum()), \'1.0 s:\', int(AV[\'10\'].sum()), \'| FLOW DATASET BUILD PASS\')\n')
open('/tmp/l3/train_flow.py','w',encoding='utf-8').write('"""Stage G: temporal U-Net -> occupancy(t) [free/static/object], ground-velocity flow (vx,vy in E_t axes), future object occupancy at +0.5 s and +1.0 s.\nFixed a priori: 60 epochs, batch 8, AdamW lr 2e-3 one-cycle, wd 1e-4, seed 0, flip-y augmentation, last-epoch model reported.\nVariants: T1 (single frame), T2, T4 (main = our_flow.pth), T4_nowarp (4 frames WITHOUT ego-motion compensation)."""\nimport json, os, sys, time\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nfrom scipy.ndimage import binary_dilation\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nDATA, FLOWD, OUT = sys.argv[1], sys.argv[2], sys.argv[3]\nos.makedirs(OUT, exist_ok=True)\ndev = \'cuda\'\nd = np.load(os.path.join(DATA, \'occ_dataset.npz\'), allow_pickle=True); g = np.load(os.path.join(FLOWD, \'flow_dataset.npz\'))\nX = torch.from_numpy(np.concatenate([d[\'XL\'], d[\'XR\']], 1)).to(dev)                    # (N,7,200,200) half\nY = torch.from_numpy(d[\'Y\'].astype(np.int64)).to(dev); split = d[\'split\']\ntr, va = np.where(split == \'train\')[0], np.where(split == \'val\')[0]\nH = torch.from_numpy(g[\'H\']).to(dev); A = torch.from_numpy(g[\'A\']).to(dev); Bv = torch.from_numpy(g[\'B\']).to(dev)\nFLOW = torch.from_numpy(g[\'FLOW\'].astype(np.float32)).to(dev); FV = torch.from_numpy(g[\'FVALID\']).to(dev)\nFUT = torch.stack([torch.from_numpy(g[\'FUT05\']), torch.from_numpy(g[\'FUT10\'])], 1).float().to(dev)       # (N,2,200,200)\nAVL = torch.stack([torch.from_numpy(g[\'AV05\']), torch.from_numpy(g[\'AV10\'])], 1).float().to(dev)          # (N,2)\nFM = np.stack([g[\'FM05\'], g[\'FM10\']], 1)\nNC = 200; xs = -50 + (np.arange(NC) + 0.5) * 0.5\nXXt, YYt = torch.meshgrid(torch.tensor(xs, dtype=torch.float32, device=dev), torch.tensor(xs, dtype=torch.float32, device=dev), indexing=\'ij\')\nEPOCHS, BS, LR = 60, 8, 2e-3\n\n\ndef stack_input(idx, T, warp):\n    """(B, T*7, 200, 200) float: frames t-(T-1)..t, history warped into the current ego frame (nearest history frame of the scene reused when short)."""\n    cur = X[idx].float(); frames = [cur]\n    for k in range(T - 1):\n        src = X[H[idx, k]].float()\n        if warp:\n            a, b = A[idx, k], Bv[idx, k]                                              # (B,2,2), (B,2)\n            qx = a[:, 0, 0, None, None] * XXt + a[:, 0, 1, None, None] * YYt + b[:, 0, None, None]\n            qy = a[:, 1, 0, None, None] * XXt + a[:, 1, 1, None, None] * YYt + b[:, 1, None, None]\n            src = F.grid_sample(src, torch.stack([qy / 50.0, qx / 50.0], -1), mode=\'bilinear\', padding_mode=\'zeros\', align_corners=False)\n        frames.append(src)\n    return torch.cat(frames[::-1], 1)\n\n\ndef block(i, o):\n    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))\n\n\nclass UNet(nn.Module):\n    def __init__(self, cin, base=32, nout=7):\n        super().__init__()\n        c = [base, base * 2, base * 4, base * 8]\n        self.e1, self.e2, self.e3, self.e4 = block(cin, c[0]), block(c[0], c[1]), block(c[1], c[2]), block(c[2], c[3])\n        self.u3, self.d3 = nn.ConvTranspose2d(c[3], c[2], 2, 2), block(c[3], c[2]); self.u2, self.d2 = nn.ConvTranspose2d(c[2], c[1], 2, 2), block(c[2], c[1])\n        self.u1, self.d1 = nn.ConvTranspose2d(c[1], c[0], 2, 2), block(c[1], c[0]); self.out = nn.Conv2d(c[0], nout, 1)\n\n    def forward(self, x):\n        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2)); e4 = self.e4(F.max_pool2d(e3, 2))\n        x = self.d3(torch.cat([self.u3(e4), e3], 1)); x = self.d2(torch.cat([self.u2(x), e2], 1)); x = self.d1(torch.cat([self.u1(x), e1], 1))\n        return self.out(x)\n\n\ncls_freq = np.array([(Y[torch.from_numpy(tr).to(dev)] == c).float().mean().item() for c in range(3)])\nwc = torch.tensor((1 / np.sqrt(cls_freq)) / (1 / np.sqrt(cls_freq)).sum() * 3, dtype=torch.float32, device=dev)\nfut_pos = float(FUT[torch.from_numpy(tr).to(dev)].mean().item()); pos_w = torch.tensor(min(30.0, (1 - fut_pos) / max(fut_pos, 1e-6) ** 1), device=dev) ** 0.5\n\n\ndef losses(out, y, flow, fv, fut, av):\n    lg = out[:, :3]; pr = F.softmax(lg, 1); oh = F.one_hot(y, 3).permute(0, 3, 1, 2).float()\n    dice = 1 - ((2 * (pr * oh).sum((0, 2, 3)) + 1) / (pr.sum((0, 2, 3)) + oh.sum((0, 2, 3)) + 1))[1:].mean()\n    l_occ = F.cross_entropy(lg, y, weight=wc) + dice\n    m = fv.float().unsqueeze(1); l_flow = (F.smooth_l1_loss(out[:, 3:5] / 1.0, flow / 10.0, reduction=\'none\') * m).sum() / (m.sum() * 2 + 1)\n    l_fut = 0\n    for j in range(2):\n        pj = torch.sigmoid(out[:, 5 + j]); tj = fut[:, j]; w_ = av[:, j].view(-1, 1, 1)\n        bce = (F.binary_cross_entropy_with_logits(out[:, 5 + j], tj, pos_weight=pos_w, reduction=\'none\') * w_).sum() / (w_.sum() * NC * NC + 1)\n        dj = 1 - (2 * (pj * tj * w_).sum() + 1) / ((pj * w_).sum() + (tj * w_).sum() + 1)\n        l_fut = l_fut + bce + dj\n    return l_occ + 2.0 * l_flow + 0.5 * l_fut, (l_occ.item(), float(l_flow), float(l_fut))\n\n\n@torch.no_grad()\ndef predict(model, T, warp, idx):\n    model.eval(); o = []\n    for i in range(0, len(idx), 8):\n        o.append(model(stack_input(torch.from_numpy(idx[i:i + 8]).to(dev), T, warp)).float().cpu())\n    o = torch.cat(o).numpy(); o[:, 3:5] *= 10.0      # the network regresses flow / 10 (m/s)\n    return o\n\n\ndef occ_metrics(pred, gt):\n    r = {}; ious = []\n    for c, nm in enumerate((\'free\', \'static\', \'object\')):\n        i = ((pred == c) & (gt == c)).sum(); u = ((pred == c) | (gt == c)).sum(); ious.append(i / max(1, u)); r[f\'IoU_{nm}\'] = round(float(i / max(1, u)), 4)\n    r[\'mIoU\'] = round(float(np.mean(ious)), 4); po, go = pred > 0, gt > 0\n    r[\'IoU_occupied\'] = round(float((po & go).sum() / max(1, (po | go).sum())), 4)\n    return r\n\n\ndef flow_metrics(fl, o_pred, gt_occ):\n    """fl (n,2,200,200) predicted, GT from the dataset (val rows). EPE / angle / false-flow definitions follow the project\'s FLOW_CONTRACT."""\n    gf = FLOW[va].cpu().numpy(); gv = FV[va].cpu().numpy(); sp = np.hypot(gf[:, 0], gf[:, 1]); pf = np.hypot(fl[:, 0], fl[:, 1])\n    mov = gv & (sp >= 0.3); sta = gv & (sp < 0.3); epe = np.hypot(fl[:, 0] - gf[:, 0], fl[:, 1] - gf[:, 1])\n    ang = np.rad2deg(np.arccos(np.clip((fl[:, 0] * gf[:, 0] + fl[:, 1] * gf[:, 1]) / np.maximum(pf * sp, 1e-6), -1, 1)))\n    am = gv & (sp >= 0.5) & (pf >= 0.1)\n    struct = gt_occ == 1\n    return dict(EPE_moving=round(float(epe[mov].mean()), 3), EPE_moving_median=round(float(np.median(epe[mov])), 3), EPE_zero_flow_baseline=round(float(sp[mov].mean()), 3),\n                angle_median_deg=round(float(np.median(ang[am])), 1) if am.any() else None, angle_coverage=round(float(am.sum() / max(1, (gv & (sp >= 0.5)).sum())), 3),\n                static_object_false_flow_mean=round(float(pf[sta].mean()), 3), static_object_false_flow_p95=round(float(np.percentile(pf[sta], 95)), 3),\n                static_structure_false_flow_mean=round(float(pf[struct].mean()), 3), n_moving_cells=int(mov.sum()), n_static_object_cells=int(sta.sum()))\n\n\ndef shift(mask, flow, h):\n    out = np.zeros_like(mask); r, c = np.where(mask)\n    if len(r) == 0:\n        return out\n    rr = np.clip(np.round(r + flow[0, r, c] * h / 0.5).astype(int), 0, NC - 1); cc = np.clip(np.round(c + flow[1, r, c] * h / 0.5).astype(int), 0, NC - 1)\n    out[rr, cc] = True\n    return out\n\n\ndef iou(a, b):\n    return float((a & b).sum() / max(1, (a | b).sum()))\n\n\ndef future_metrics(out, occ_pred, fl):\n    res = {}\n    gt_occ = Y[va].cpu().numpy(); gf = FLOW[va].cpu().numpy(); gv = FV[va].cpu().numpy()\n    for j, h in enumerate((0.5, 1.0)):\n        av = AVL[va][:, j].cpu().numpy() > 0; fut = FUT[va][:, j].cpu().numpy() > 0; fm = FM[va][:, j] > 0; ids = np.where(av)[0]\n        prob = 1 / (1 + np.exp(-out[:, 5 + j])); head = prob > 0.5\n        pred_obj = occ_pred == 2; gt_obj = gt_occ == 2; spd = np.hypot(fl[:, 0], fl[:, 1])\n        gsp = np.hypot(gf[:, 0], gf[:, 1])\n        r = dict(n_samples=int(len(ids)))\n        r[\'IoU_direct_future_head\'] = round(float(np.mean([iou(head[i], fut[i]) for i in ids])), 4)\n        r[\'IoU_persistence_predicted_objects\'] = round(float(np.mean([iou(pred_obj[i], fut[i]) for i in ids])), 4)\n        r[\'IoU_persistence_GT_objects(oracle)\'] = round(float(np.mean([iou(gt_obj[i], fut[i]) for i in ids])), 4)\n        r[\'IoU_flow_extrapolation_GT_objects+GT_flow(oracle)\'] = round(float(np.mean([iou(np.where(gv[i] & (gsp[i] >= 0.3), False, gt_obj[i]) | shift(gv[i] & (gsp[i] >= 0.3), gf[i], h), fut[i]) for i in ids])), 4)\n        r[\'IoU_flow_extrapolation_predicted\'] = round(float(np.mean([iou(np.where(pred_obj[i] & (spd[i] >= 0.3), False, pred_obj[i]) | shift(pred_obj[i] & (spd[i] >= 0.3), fl[i], h), fut[i]) for i in ids])), 4)\n        pm = pred_obj & (spd >= 0.3)      # moving-object footprint (project\'s flow-warped metric): shifted vs not shifted, against the moving GT footprint at t+h\n        r[\'moving_IoU_shifted_by_predicted_flow\'] = round(float(np.mean([iou(shift(pm[i], fl[i], h), fm[i]) for i in ids if fm[i].any()])), 4)\n        r[\'moving_IoU_not_shifted\'] = round(float(np.mean([iou(pm[i], fm[i]) for i in ids if fm[i].any()])), 4)\n        r[\'n_samples_with_moving_objects\'] = int(sum(fm[i].any() for i in ids))\n        res[f\'{h}s\'] = r\n    return res\n\n\nVAR = {\'T1\': (1, True), \'T2\': (2, True), \'T4\': (4, True), \'T4_nowarp\': (4, False)}\nresults, logs, saved = {}, {}, {}\nYva = Y[va].cpu().numpy()\nfor name, (T, warp) in VAR.items():\n    torch.manual_seed(0); np.random.seed(0)\n    model = UNet(T * 7).to(dev); npar = sum(p.numel() for p in model.parameters())\n    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4); sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * int(np.ceil(len(tr) / BS)), pct_start=0.1)\n    log = []; t0 = time.time()\n    for ep in range(1, EPOCHS + 1):\n        model.train(); perm = np.random.permutation(tr); tot = 0\n        for i in range(0, len(perm), BS):\n            b = torch.from_numpy(perm[i:i + BS]).to(dev); x = stack_input(b, T, warp); y = Y[b]; fl = FLOW[b]; fv = FV[b]; fut = FUT[b]; av = AVL[b]\n            if np.random.rand() < 0.5:\n                x = torch.flip(x, [3]); y = torch.flip(y, [2]); fl = torch.flip(fl, [3]) * torch.tensor([1., -1.], device=dev).view(1, 2, 1, 1); fv = torch.flip(fv, [2]); fut = torch.flip(fut, [3])\n            out = model(x); loss, parts = losses(out.float(), y, fl, fv, fut, av)\n            opt.zero_grad(); loss.backward(); opt.step(); sch.step(); tot += loss.item() * len(b)\n        rec = dict(epoch=ep, train_loss=tot / len(tr))\n        if ep % 10 == 0 or ep == 1:\n            o = predict(model, T, warp, va); rec[\'val_mIoU\'] = occ_metrics(o[:, :3].argmax(1), Yva)[\'mIoU\']; rec[\'val_EPE_moving\'] = flow_metrics(o[:, 3:5], None, Yva)[\'EPE_moving\']\n        log.append(rec)\n        if ep % 20 == 0:\n            print(name, rec, flush=True)\n    secs = time.time() - t0\n    path = os.path.join(OUT, f\'our_flow_{name}.pth\'); torch.save(dict(state_dict=model.state_dict(), T=T, warp=warp, variant=name, outputs=\'0-2 occupancy logits (free/static/object), 3-4 flow vx,vy m/s (ground velocity, ego(t) axes), 5-6 future object logits +0.5 s / +1.0 s\',\n                                                                        note=f\'Stage G {name}: trained on nuScenes-mini TRAIN only, {EPOCHS} epochs\'), path)\n    o1 = predict(model, T, warp, va); ck = torch.load(path, map_location=dev); m2 = UNet(T * 7).to(dev); m2.load_state_dict(ck[\'state_dict\']); o2 = predict(m2, T, warp, va)\n    reload_ok = bool(np.allclose(o1, o2, atol=1e-4))\n    occ_pred = o1[:, :3].argmax(1); fl = o1[:, 3:5]\n    xb = stack_input(torch.from_numpy(va[:1]).to(dev), T, warp); model.eval()\n    with torch.no_grad():\n        for _ in range(10): model(xb)\n        torch.cuda.synchronize(); t1 = time.time()\n        for _ in range(100): model(xb)\n        torch.cuda.synchronize(); lat = (time.time() - t1) / 100\n    results[name] = dict(T=T, ego_motion_compensated=warp, occupancy=occ_metrics(occ_pred, Yva), flow=flow_metrics(fl, occ_pred, Yva), future=future_metrics(o1, occ_pred, fl),\n                         params_M=round(npar / 1e6, 2), train_secs=round(secs), reload_identical=reload_ok, latency_ms_T4=round(lat * 1000, 2), final_train_loss=round(log[-1][\'train_loss\'], 4))\n    logs[name] = log; saved[name] = o1.astype(np.float16)\n    print(name, json.dumps(results[name])[:1400], flush=True)\njson.dump(dict(results=results, logs=logs, epochs=EPOCHS, batch=BS, lr=LR, n_train=int(len(tr)), n_val=int(len(va))), open(os.path.join(OUT, \'flow_metrics.json\'), \'w\'), indent=2)\nnp.savez_compressed(os.path.join(OUT, \'flow_val_predictions.npz\'), **{f\'out_{k}\': v for k, v in saved.items()}, gt_occ=Yva.astype(np.uint8), gt_flow=FLOW[va].cpu().numpy().astype(np.float16), gt_flow_valid=FV[va].cpu().numpy(),\n                    fut05=FUT[va][:, 0].cpu().numpy() > 0, fut10=FUT[va][:, 1].cpu().numpy() > 0, av05=AVL[va][:, 0].cpu().numpy() > 0, av10=AVL[va][:, 1].cpu().numpy() > 0, token=d[\'token\'][va], scene=d[\'scene\'][va])\nfig, ax = plt.subplots(1, 2, figsize=(13, 4.5))\nfor name, log in logs.items():\n    ax[0].plot([r[\'epoch\'] for r in log], [r[\'train_loss\'] for r in log], label=name)\n    e = [r for r in log if \'val_EPE_moving\' in r]; ax[1].plot([r[\'epoch\'] for r in e], [r[\'val_EPE_moving\'] for r in e], \'o-\', label=name)\nax[0].set_yscale(\'log\'); ax[0].set_title(\'train loss\'); ax[0].legend(); ax[1].set_title(\'mini VAL flow EPE on moving cells (monitoring only)\'); ax[1].legend()\nfig.tight_layout(); fig.savefig(os.path.join(OUT, \'flow_training_curves.png\'), dpi=100)\nprint(\'FLOW TRAIN PASS\')\n')
print('written')

In [ ]:
%%bash
set -euo pipefail
export W=/tmp/l3 MPLBACKEND=Agg CUDA_VISIBLE_DEVICES=0
mkdir -p /kaggle/working/outputs $W/data
TAR=$(find /kaggle/input -maxdepth 6 -name 'env_bevfusion.tar.*' | head -1); echo "$TAR"
case "$TAR" in *.zst) zstd -dc "$TAR" | tar -C $W -xf - envs/bevfusion ;; *.gz) tar -C $W -xzf "$TAR" envs/bevfusion ;; esac
PY=$W/envs/bevfusion/bin/python
$PY -c "import torch; print('torch', torch.__version__, torch.cuda.get_device_name(0))"
$PY -W ignore $W/build_occ_dataset.py $W/data 2>&1 | tail -4
$PY -W ignore $W/build_flow_dataset.py $W/data 2>&1 | tail -6
$PY -W ignore $W/train_flow.py $W/data $W/data /kaggle/working/outputs 2>&1 | tail -30
ls -la /kaggle/working/outputs
